# Đọc dữ liệu

In [25]:
# ==============================================================================
# CELL 1: IMPORT THƯ VIỆN VÀ ĐỌC DỮ LIỆU SẠCH (ĐÃ CHIA TRAIN/TEST TỪ TRƯỚC)
# ==============================================================================
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# 1. Đọc thẳng 2 file Train và Test đã lưu ở bước Clean
df_train_clean = pd.read_csv('data_output/1_data_cleaned_train.csv')
df_test_clean = pd.read_csv('data_output/1_data_cleaned_test.csv')

print("✅ Đã tải dữ liệu Train và Test thành công!")

# 2. Tách lại Biến độc lập (X) và Biến mục tiêu (y)
# LƯU Ý: Chỉnh sửa 'Khoảng giá' thành tên cột mục tiêu chính xác của bạn nếu cần
target_col = 'Khoảng giá' 

X_train = df_train_clean.drop(columns=[target_col])
y_train = df_train_clean[target_col]

X_test = df_test_clean.drop(columns=[target_col])
y_test = df_test_clean[target_col]

print(f"Kích thước X_train: {X_train.shape}")
print(f"Kích thước X_test: {X_test.shape}")

✅ Đã tải dữ liệu Train và Test thành công!
Kích thước X_train: (4774, 18)
Kích thước X_test: (1197, 18)


# Encode các biến catergorical: 'Pháp lý', 'Nội thất' và 'Loại đường vào.

In [26]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: ONE-HOT ENCODING (PHÁP LÝ & NỘI THẤT)
# ═══════════════════════════════════════════════════════════════════════
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# 1. Khởi tạo bộ One-Hot Encoder
# - drop='first': Bỏ cột đầu tiên của mỗi thuộc tính để tránh bẫy đa cộng tuyến toán học (Dummy Variable Trap)
# - handle_unknown='ignore': Nếu tập Test có nhãn lạ, tự động điền toàn bộ số 0 thay vì báo lỗi
ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

# Các cột cần One-Hot
ohe_cols = ['Pháp lý', 'Nội thất']

# 2. Học cấu trúc từ tập Train và biến đổi tập Train
train_ohe = ohe.fit_transform(X_train[ohe_cols])

# 3. Chỉ biến đổi tập Test dựa trên những gì đã học từ Train
test_ohe = ohe.transform(X_test[ohe_cols])

# 4. Lấy tên các cột mới sau khi mã hóa (Ví dụ: Pháp lý_Sổ đỏ, Nội thất_Đầy đủ...)
encoded_col_names = ohe.get_feature_names_out(ohe_cols)

# 5. Chuyển mảng Numpy thu được thành DataFrame để dễ ghép nối
df_train_ohe = pd.DataFrame(train_ohe, columns=encoded_col_names, index=X_train.index)
df_test_ohe = pd.DataFrame(test_ohe, columns=encoded_col_names, index=X_test.index)

# 6. Ghép các cột mới gộp vào dữ liệu gốc và xóa bỏ 2 cột chữ ban đầu
X_train = pd.concat([X_train.drop(columns=ohe_cols), df_train_ohe], axis=1)
X_test = pd.concat([X_test.drop(columns=ohe_cols), df_test_ohe], axis=1)

print("✅ Đã áp dụng One-Hot Encoding thành công cho Pháp lý và Nội thất!")
print(f"Các cột mới được sinh ra: {list(encoded_col_names)}")

✅ Đã áp dụng One-Hot Encoding thành công cho Pháp lý và Nội thất!
Các cột mới được sinh ra: ['Pháp lý_Sổ đỏ', 'Pháp lý_không có', 'Nội thất_Cơ bản', 'Nội thất_Trống / Nhà thô', 'Nội thất_Đầy đủ']


In [27]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: ORDINAL ENCODING (LOẠI ĐƯỜNG VÀO)
# ═══════════════════════════════════════════════════════════════════════
from sklearn.preprocessing import OrdinalEncoder

# 1. Định nghĩa thứ tự phân bậc rõ ràng từ thấp đến cao theo logic thực tế
# Nhóm 'Chưa xác định' ép về bậc thấp nhất để giữ mạch tăng đơn điệu cho 3 nhóm còn lại
categories_order = [['Chưa xác định', 'Hẻm xe máy', 'Hẻm ô tô', 'Mặt tiền']]

# 2. Khởi tạo bộ Ordinal Encoder với danh sách thứ tự đã ép sẵn
# handle_unknown='use_encoded_value', unknown_value=-1: Phòng hờ nếu tập Test có giá trị lạ
ordinal_enc = OrdinalEncoder(categories=categories_order, 
                             handle_unknown='use_encoded_value', 
                             unknown_value=-1)

# Cột cần mã hóa thứ bậc
ord_col = ['Loại đường vào']

# 3. Tiến hành Fit và Transform trên tập Train
X_train[ord_col] = ordinal_enc.fit_transform(X_train[ord_col])

# 4. Chỉ Transform trên tập Test
X_test[ord_col] = ordinal_enc.transform(X_test[ord_col])

print("✅ Đã áp dụng Ordinal Encoding thành công cho cột 'Loại đường vào'!")
print("Thứ tự ánh xạ tương ứng: Chưa xác định -> 0, Hẻm xe máy -> 1, Hẻm ô tô -> 2, Mặt tiền -> 3")

✅ Đã áp dụng Ordinal Encoding thành công cho cột 'Loại đường vào'!
Thứ tự ánh xạ tương ứng: Chưa xác định -> 0, Hẻm xe máy -> 1, Hẻm ô tô -> 2, Mặt tiền -> 3


In [28]:
# ═══════════════════════════════════════════════════════════════════════
# CELL LƯU FILE 2: DỮ LIỆU DÀNH CHO MÔ HÌNH DẠNG CÂY (TREE-BASED MODELS)
# ═══════════════════════════════════════════════════════════════════════

# Tại đây, Pháp lý, Nội thất, Loại đường vào ĐÃ LÀ SỐ, nhưng Diện tích, Đường vào... CHƯA SCALE
# Tạo một bản sao sâu (deep copy) để lưu lại trạng thái này trước khi bị gán đè scale ở các cell sau
X_train_tree = X_train.copy()
X_test_tree = X_test.copy()

# Ghép với target gốc (Mô hình cây chạy tốt trên cả target gốc lẫn target log)  
df_train_tree = pd.concat([X_train_tree, y_train], axis=1)
df_test_tree = pd.concat([X_test_tree, y_test], axis=1) if 'y_test' in locals() else X_test_tree.copy()

# Xuất file
df_train_tree.to_csv('data_output/2_data_preprocessed_tree_train.csv', index=False)
df_test_tree.to_csv('data_output/2_data_preprocessed_tree_test.csv', index=False)

print("💾 FILE 2: Đã lưu dữ liệu cho mô hình dạng Cây thành công!")
print("   -> data_output/2_data_preprocessed_tree_train.csv")
print("   -> data_output/2_data_preprocessed_tree_test.csv")

💾 FILE 2: Đã lưu dữ liệu cho mô hình dạng Cây thành công!
   -> data_output/2_data_preprocessed_tree_train.csv
   -> data_output/2_data_preprocessed_tree_test.csv


In [29]:
print(X_train.columns)

Index(['Diện tích', 'Số tầng', 'Đường vào', 'Latitude', 'Longitude',
       'Loại đường vào', 'hem_xe_hoi', 'gan_cho_sieu_thi', 'gan_truong_hoc',
       'gan_benh_vien', 'gan_cong_vien_ho_nuoc', 'Đường', 'Phường/Xã/Thị Trấn',
       'Quận', 'duong_vao_null_flag', 'Tổng số phòng', 'Pháp lý_Sổ đỏ',
       'Pháp lý_không có', 'Nội thất_Cơ bản', 'Nội thất_Trống / Nhà thô',
       'Nội thất_Đầy đủ'],
      dtype='object')
